# Earlier-fold training (RB weekly out-of-sample test) — Kaggle: T4 ×2 + Internet ON

**Run All, walk away (~40 min).** Trains 9 baseline v1 runs — folds through **2019 / 2020 / 2021** ×
seeds 42/43/44 — so the running-back weekly edge (found on 2023-25) can get a **clean out-of-sample**
test on 2020/2021/2022 (seasons disjoint from the discovery set).

This is the ONLY promising untested lead. Everything else this project measured says season-long draft
ranking is not the model's edge; weekly RB is the one place it might be. Nothing here touches the live site.

1. Session options → Accelerator **GPU T4 ×2** (not the P100 — sm_60, unsupported), **Internet ON**.
2. Run All.
3. Come back to a push confirmation or a `rb_oos_artifacts.zip` to download and commit locally.

Resumable after a cutoff (skip-if-complete). The out-of-sample eval is a **separate** local session.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:  # fresh Kaggle/Colab session: not inside a clone yet, so make one
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
import subprocess

# 9 runs: baseline v1 at earlier folds x seeds 42/43/44.
# seed 42 = config default (no --seed) -> models/transformer/v1/through{2019,2020,2021}
# seeds 43/44 -> v1_s43 / v1_s44. skip-if-complete makes a re-run resume-only.
configs = ["configs/transformer_v1_through2019.yaml",
           "configs/transformer_v1_through2020.yaml",
           "configs/transformer_v1_through2021.yaml"]
for seed in (42, 43, 44):
    for cfg in configs:
        cmd = ["python", "-m", "ffmodel.model.train", "--config", cfg,
               "--features-parquet", "data/features_2012_2025.parquet"]
        if seed != 42:
            cmd += ["--seed", str(seed)]
        print()
        print(f"=== seed {seed}  {cfg} ===", flush=True)
        subprocess.run(cmd, check=True)
print()
print("All 9 earlier-fold runs complete.")

In [ ]:
import subprocess, zipfile
from pathlib import Path

# Commit the earlier-fold artifacts. Nothing deploys; ARTIFACT_ROOT is unchanged.
# through{2019,2020,2021} land under the SAME v1 / v1_s43 / v1_s44 roots as the
# existing folds, so add those roots (git only stages the new through* dirs).
roots = [f"models/transformer/v1{s}" for s in ("", "_s43", "_s44")]
subprocess.run(["git", "add", *roots], check=False)
subprocess.run(["git", "commit", "-m",
                "model: v1 earlier folds (through2019/2020/2021) for RB out-of-sample test"],
               check=False)
pushed = subprocess.run(["git", "push"], check=False).returncode == 0
if pushed:
    print("Pushed earlier-fold artifacts. Ready for the RB out-of-sample eval.")
else:
    zp = Path("rb_oos_artifacts.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for r in roots:
            for f in Path(r).rglob("through201*/*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
    for line in [
        f"git push failed -- wrote {zp.resolve()} instead.",
        "Download it, then in your local clone:",
        f"  unzip {zp.name}",
        "  git add models/transformer/v1 models/transformer/v1_s43 models/transformer/v1_s44",
        '  git commit -m "model: v1 earlier folds (through2019/2020/2021)"',
        "  git push",
    ]:
        print(line)